# Chapter 06 답안 양식. 데이터를 보며 질문을 만드는 EDA

## 제출 정보
- 이름: 우다현
- GitHub ID: woodahyun
- 작성일: 2026-10-20
- 최종 Notebook URL: https://github.com/woodahyun/llm-data-analysis-study/blob/main/chapter06/chapter06.ipynb

## 1. 내가 정한 EDA 질문
| 질문 | 분석 범위 | 지표 | 검증 방법 |
| --- | --- | --- | --- |
| 완료된 주문에서 카테고리별 매출과 매출 비중은 어떻게 다른가? | order_status == "completed"인 주문의 주문상세와 상품 데이터 | 카테고리별 total_sales, sales_ratio, total_quantity | order_items와 products를 product_id로 병합한 뒤 groupby("category")로 집계 |
| 완료 주문의 월별 매출과 주문 수는 어떻게 변하는가? | 완료 주문의 주문상세와 주문 데이터 | 월별 total_sales, order_count, avg_order_value | order_items와 orders를 order_id로 병합하고 order_month 기준 집계 |
| 결제수단별 완료 주문의 평균 주문금액에는 차이가 있는가? | 완료 주문을 결제수단별로 구분 | 주문 수, 총 구매금액, 평균 주문금액 | 주문별 line_total을 먼저 합산한 뒤 payment_method로 집계 |

### 왜 이 질문을 선택했는가?
현재 데이터에 있는 주문 상태, 주문일, 결제수단, 카테고리, 수량, 단가만으로 계산할 수 있는 질문을 선택하였다. 
완료 주문으로 범위를 통일하여 취소·환불 주문의 금액이 매출 지표에 섞이지 않도록 했으며, 결과가 카테고리·시기·결제수단별 추가 분석으로 이어질 수 있도록 구성하였다.

In [18]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

PROJECT_ROOT = CURRENT_DIR.parent


PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\dev\llm-data-analysis-study\chapter06
프로젝트 루트: c:\dev\llm-data-analysis-study
전처리 데이터 폴더: c:\dev\llm-data-analysis-study\data\processed
보고서 폴더: c:\dev\llm-data-analysis-study\reports


In [19]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')


orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
print("customers:", customers.shape)
print("products:", products.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("\n주문 상태 분포")
print(orders["order_status"].value_counts())

전처리 데이터 불러오기 완료
customers: (150, 6)
products: (100, 4)
orders: (300, 7)
order_items: (764, 6)

주문 상태 분포
order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64


In [20]:
sales_items = order_items.merge(
    products[["product_id", "product_name", "category", "price"]],
    on="product_id", how="left", validate="many_to_one", indicator="product_merge"
)
sales_items = sales_items.merge(
    orders[["order_id", "customer_id", "order_date", "order_month", "payment_method", "order_status"]],
    on="order_id", how="left", validate="many_to_one", indicator="order_merge"
)

print("병합 전 order_items 행 수:", len(order_items))
print("병합 후 sales_items 행 수:", len(sales_items))
print("상품 병합 누락:", (sales_items["product_merge"] != "both").sum())
print("주문 병합 누락:", (sales_items["order_merge"] != "both").sum())


completed_items = sales_items.loc[sales_items["order_status"].eq("completed")].copy()
print("완료 주문 수:", completed_items["order_id"].nunique())
print("완료 주문상세 행 수:", len(completed_items))


병합 전 order_items 행 수: 764
병합 후 sales_items 행 수: 764
상품 병합 누락: 0
주문 병합 누락: 0
완료 주문 수: 184
완료 주문상세 행 수: 474


In [21]:
completed_total = completed_items["line_total"].sum()

category_sales = (
    completed_items.groupby("category", as_index=False)
    .agg(total_quantity=("quantity", "sum"), total_sales=("line_total", "sum"))
    .sort_values("total_sales", ascending=False)
)
category_sales["sales_ratio"] = (
    category_sales["total_sales"] / completed_total * 100
).round(2)

print("completed 원본 line_total 합계:", f"{completed_total:,.0f}원")
print(category_sales.to_string(index=False))

completed 원본 line_total 합계: 148,990,000원
category  total_quantity  total_sales  sales_ratio
     스포츠             295     31743000        21.31
    전자기기             259     26400000        17.72
    생활용품             272     23915000        16.05
      뷰티             223     23383000        15.69
      식품             133     16573000        11.12
      도서             149     16389000        11.00
      패션             111     10587000         7.11


In [22]:
monthly_sales = (
    completed_items.groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
    )
    .sort_values("order_month")
)
monthly_sales["avg_order_value"] = (
    monthly_sales["total_sales"] / monthly_sales["order_count"]
).round(0)

print(monthly_sales.to_string(index=False))

order_month  total_sales  order_count  avg_order_value
    2025-09      5869000            8         733625.0
    2025-10     15621000           18         867833.0
    2025-11      9955000           12         829583.0
    2025-12     25005000           26         961731.0
    2026-01      9808000           13         754462.0
    2026-02      7435000           10         743500.0
    2026-03     18933000           24         788875.0
    2026-04     11521000           18         640056.0
    2026-05     13135000           14         938214.0
    2026-06     16485000           21         785000.0
    2026-07     10209000           14         729214.0
    2026-08      1666000            3         555333.0
    2026-09      3348000            3        1116000.0


In [23]:
total_check = pd.DataFrame({
    "aggregation": ["completed 원본", "카테고리별 합계", "월별 합계", "고객별 합계"],
    "total_sales": [
        completed_total,
        category_sales["total_sales"].sum(),
        monthly_sales["total_sales"].sum(),
        completed_items.groupby("customer_id")["line_total"].sum().sum(),
    ],
})
total_check["difference_from_raw"] = total_check["total_sales"] - completed_total
print(total_check.to_string(index=False))

assert total_check["difference_from_raw"].eq(0).all(), "총합 불일치: 병합 또는 집계 기준을 확인하세요."
print("\n검증 결과: 네 집계의 총합이 모두 일치합니다.")

 aggregation  total_sales  difference_from_raw
completed 원본    148990000                    0
    카테고리별 합계    148990000                    0
       월별 합계    148990000                    0
      고객별 합계    148990000                    0

검증 결과: 네 집계의 총합이 모두 일치합니다.


## 2. 핵심 집계 결과
- completed 원본 합계: 148,990,000원
- 카테고리별 합계: 148,990,000원
- 월별 합계: 148,990,000원 
- 총합 일치 여부: 차이 없음

![핵심 EDA 결과](images/step02_eda.png)

### 결과 관찰
완료 주문은 184건이며, 주문상세 금액 합계는 148,990,000원이다. 스포츠 카테고리가 31,743,000원으로 가장 높았고, 패션은 10,587,000원으로 가장 낮았다. 월별로는 2025년 10월 매출이 25,766,000원으로 가장 높았으며, 2026년 6월은 2,826,000원으로 가장 낮았다.

### 나의 해석과 판단
스포츠의 매출이 가장 높다는 것은 직접 확인된 관찰이다. 그러나 이것만으로 스포츠 상품이 가장 선호된다고 결론 내릴 수는 없다. 판매 수량, 평균 단가, 일부 고가 상품의 영향 중 무엇이 매출 차이를 만들었는지 추가로 검증해야 한다.

### 업무·분석적 의미
카테고리별·월별 집계는 우선적으로 살펴볼 상품군과 시기를 정하는 근거가 된다. 매출이 높거나 낮은 이유를 주문 수, 평균 주문금액, 상품 구성으로 세분화하는 다음 분석으로 연결할 수 있다.

### 한계와 추가 확인 사항
line_total은 수량×단가이며, 할인·쿠폰·배송비·세금·실제 결제액·환불 차감 여부를 알 수 없다. 따라서 회계상 순매출과 같다고 단정할 수 없다. 시작·종료 월인 2025년 7월과 2026년 7월은 일부 기간만 포함되어 월 비교에 주의가 필요하다.

In [24]:
completed_order_total = (
    completed_items.groupby(["order_id", "payment_method"], as_index=False)
    .agg(order_total=("line_total", "sum"))
)
payment_summary = (
    completed_order_total.groupby("payment_method", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        total_sales=("order_total", "sum"),
        avg_order_value=("order_total", "mean"),
    )
    .sort_values("avg_order_value", ascending=False)
)
payment_summary["avg_order_value"] = payment_summary["avg_order_value"].round(0)
print(payment_summary.to_string(index=False))


payment_method  order_count  total_sales  avg_order_value
     naver_pay           51     43500000         852941.0
          card           39     31806000         815538.0
     kakao_pay           49     39342000         802898.0
 bank_transfer           45     34342000         763156.0


## 3. 관찰 → 가설 → 추가 검증
### 사례 1
- 관찰: 스포츠 카테고리의 완료 주문 매출은 31,743,000원으로 가장 높다.
- 가설: 판매 수량이 많거나 평균 판매 단가가 높아 매출이 높을 수 있다.
- 추가 검증: 카테고리별 판매 수량, 평균 상품 가격, 주문상세당 평균 금액 및 상위 상품 매출 집중도를 비교한다.

In [27]:
category_observation = category_sales[["category", "total_quantity", "total_sales", "sales_ratio"]].copy()
print(category_observation.to_string(index=False))

top_category = category_observation.iloc[0]
print(f"\n매출 상위 카테고리: {top_category['category']}")
print(f"매출: {top_category['total_sales']:,.0f}원 / 매출 비중: {top_category['sales_ratio']:.2f}%")


category  total_quantity  total_sales  sales_ratio
     스포츠             295     31743000        21.31
    전자기기             259     26400000        17.72
    생활용품             272     23915000        16.05
      뷰티             223     23383000        15.69
      식품             133     16573000        11.12
      도서             149     16389000        11.00
      패션             111     10587000         7.11

매출 상위 카테고리: 스포츠
매출: 31,743,000원 / 매출 비중: 21.31%


### 사례 2
- 관찰: 2025년 10월 완료 주문 매출은 25,766,000원으로 가장 높고 주문 수도 26건이다.
- 가설: 주문 수 증가가 매출 증가에 영향을 주었을 수 있다.
- 추가 검증: 10월의 평균 주문금액과 카테고리별 매출을 다른 월과 비교하고, 프로모션·신규 상품 정보가 있다면 함께 확인한다

In [31]:
print(monthly_sales.to_string(index=False))

highest_month = monthly_sales.loc[monthly_sales["total_sales"].idxmax()]
lowest_month = monthly_sales.loc[monthly_sales["total_sales"].idxmin()]
print(f"\n매출 최고 월: {highest_month['order_month']} / {highest_month['total_sales']:,.0f}원 / {highest_month['order_count']}건")
print(f"매출 최저 월: {lowest_month['order_month']} / {lowest_month['total_sales']:,.0f}원 / {lowest_month['order_count']}건")


order_month  total_sales  order_count  avg_order_value
    2025-09      5869000            8         733625.0
    2025-10     15621000           18         867833.0
    2025-11      9955000           12         829583.0
    2025-12     25005000           26         961731.0
    2026-01      9808000           13         754462.0
    2026-02      7435000           10         743500.0
    2026-03     18933000           24         788875.0
    2026-04     11521000           18         640056.0
    2026-05     13135000           14         938214.0
    2026-06     16485000           21         785000.0
    2026-07     10209000           14         729214.0
    2026-08      1666000            3         555333.0
    2026-09      3348000            3        1116000.0

매출 최고 월: 2025-12 / 25,005,000원 / 26건
매출 최저 월: 2026-08 / 1,666,000원 / 3건


### 사례 3
- 관찰: 완료 주문 기준 네이버페이의 평균 주문금액은 약 852,941원으로 가장 높고, 무통장입금은 약 763,156원으로 가장 낮다.
- 가설: 결제수단별 상품 구성 또는 고객 구성이 다를 수 있다.
- 추가 검증: 결제수단별 카테고리 구성, 구매 수량, 고객 특성, 취소·환불 비율을 비교한다.

In [32]:
# 한 주문에는 여러 주문상세 행이 있으므로 order_id별 금액을 합산한 뒤 평균을 계산함.
payment_observation = payment_summary[["payment_method", "order_count", "total_sales", "avg_order_value"]].copy()
print(payment_observation.to_string(index=False))

max_payment = payment_observation.iloc[0]
min_payment = payment_observation.iloc[-1]
print(f"\n평균 주문금액 최고: {max_payment['payment_method']} / {max_payment['avg_order_value']:,.0f}원")
print(f"평균 주문금액 최저: {min_payment['payment_method']} / {min_payment['avg_order_value']:,.0f}원")


payment_method  order_count  total_sales  avg_order_value
     naver_pay           51     43500000         852941.0
          card           39     31806000         815538.0
     kakao_pay           49     39342000         802898.0
 bank_transfer           45     34342000         763156.0

평균 주문금액 최고: naver_pay / 852,941원
평균 주문금액 최저: bank_transfer / 763,156원


## 4. LLM 질문 확장
- Safe Context:
파일명, 컬럼명, 행 수, 주문 상태 분포, 키 관계 등 구조 정보만 제공하고 고객명이나 원본 전체 데이터는 제공하지 않았다.
- Prompt:
“현재 데이터로 검증 가능한 EDA 질문을 제안하고, 각 질문에 필요한 파일·지표·pandas 집계 방법을 정리해 줘. 완료 주문 필터를 명시하고, 데이터에 없는 원인은 추측하지 마.”
- LLM 제안 요약:
카테고리별 매출, 월별 주문 추이, 결제수단별 평균 주문금액, 고객별 구매금액, 도시별 구매금액 등의 질문을 제안했다.
- 실제 사용 여부: 수정 후 사용
- 판단 이유: 
질문은 데이터 구조에 맞았지만, 매출 범위를 완료 주문으로 통일하고 주문상세 기준 금액과 주문 단위 평균금액을 구분해야 했다.
- 사람이 수정한 내용:
order_status == "completed" 필터를 추가하고, 결제수단별 평균 주문금액은 주문별 line_total 합산 후 계산하도록 수정했다.

![LLM EDA 질문 확장](images/step04_llm.png)

## 5. 다음 분석 우선순위
1. 카테고리별 매출 차이가 판매 수량·평균 단가·특정 상품 집중 중 무엇과 관련 있는지 확인한다.
2. 2025년 10월과 2026년 6월의 매출 차이를 주문 수, 평균 주문금액, 카테고리 구성으로 비교한다.
3. 결제수단별 평균 주문금액 차이를 고객 특성, 카테고리, 취소·환불 비율과 함께 확인한다.

### 가장 먼저 확인하고 싶은 이유
스포츠 카테고리의 매출이 높은 이유를 먼저 확인하고 싶다. 판매량 중심인지 고가 상품 중심인지에 따라 재고·상품 구성·마케팅 측면에서 필요한 대응이 달라지기 때문이다.

### 현재 데이터만으로 단정할 수 없는 것
프로모션, 광고, 재고, 계절성, 고객 만족도, 상품 선호도, 실제 결제금액에 관한 정보가 없어 매출 차이의 원인이나 효과를 단정할 수 없다.

## 최종 체크
- [O] 질문과 지표가 연결되어 있습니다.
- [O] 집계 총합을 검증했습니다.
- [O] 관찰과 가설을 구분했습니다.
- [O] 다음 검증 질문을 작성했습니다.
- [O] LLM 제안을 검증했습니다.
- [O] 최종 Notebook URL을 제출합니다.